## Limpieza de cronorunner.com

A diferencia de RaceResult y youevent, `scrape_cronorunner.ipynb` ya entrega casi todo hecho: fecha, municipio y el recuento real de finishers por sexo (sacado del gráfico Chart.js de cada página de resultados, no estimado). Así que esta limpieza es más corta -- el trabajo que queda es:

1. Construir `nombre_carrera`: el catálogo trae una fila por **prueba** dentro de cada día de carrera (p.ej. el mismo día "XXII Volta a peu Loriguilla" tiene una prueba de 10k para adultos y varias carreras infantiles por categoría de edad). Cada prueba es, a efectos de análisis, una carrera distinta -- con su propia distancia y su propio público -- así que mantenemos esa granularidad y añadimos la modalidad al nombre cuando hay más de una prueba ese día.
2. `distancia` en km (la fuente ya trae metros exactos).
3. `dia_semana` a partir de la fecha.
4. `tipo_modalidad` y `publico`, clasificados por palabras clave (igual que en youevent/raceresult, pero aquí con más pistas porque el nombre de la prueba suele decir la categoría explícitamente: "Benjamín", "Cadete", "10k"...).
5. `comarca`/`provincia`, geocodificando `municipio` -- que aquí ya es un nombre de población limpio (lo extrae el propio scraper), no hay que adivinarlo dentro del nombre del evento como en championsxip/youevent, así que debería salir más fiable.

Todas las funciones de más abajo se prueban al final del notebook contra una muestra real de 12 filas (dos días de carrera reales, los mismos que usa `scrape_cronorunner.ipynb` para probar el scraper) antes de fiarnos de ellas con el catálogo completo.

In [1]:
import re
import pandas as pd

RUTA_SUCIO = "../../data/raw/cronorunner/DF_CRONORUNNER_SUCIO.csv"
df = pd.read_csv(RUTA_SUCIO, dtype={"evento_ref": str, "resultado_ref": str})
print(df.shape)
df.head()


(3145, 9)


,evento_ref,resultado_ref,nombre_evento,nombre_prueba,fecha,municipio,distancia_m,finisher_h,finisher_d
0,448-2212,448-2212-3895,XXII VOLTA A PEU LORIGUILLA,Junior (2009-2010),2026-09-04,Loriguilla,0,NaN,NaN
1,448-2212,448-2212-3861,XXII VOLTA A PEU LORIGUILLA,Benjamin (2017-2018),2026-09-04,Loriguilla,300,8.0,9.0
2,448-2212,448-2212-3862,XXII VOLTA A PEU LORIGUILLA,Alevin (2015-2016),2026-09-04,Loriguilla,700,7.0,5.0
3,448-2212,448-2212-3863,XXII VOLTA A PEU LORIGUILLA,Infantil (2013-2014),2026-09-04,Loriguilla,1050,1.0,2.0
4,448-2212,448-2212-3864,XXII VOLTA A PEU LORIGUILLA,Cadete (2011-2012),2026-09-04,Loriguilla,1400,2.0,1.0


### Funciones de limpieza

Cuando un día de carrera tiene una sola prueba, el nombre del evento ya es el nombre de la carrera. Cuando tiene varias (lo normal: la prueba absoluta + las infantiles por categoría), añadimos el nombre de la prueba entre paréntesis para no perder esa información y para que cada fila tenga un nombre único.

In [2]:
def construir_nombre_carrera(df_in: pd.DataFrame) -> pd.Series:
    n_pruebas_por_evento = df_in.groupby("evento_ref")["resultado_ref"].transform("nunique")
    nombre_carrera = df_in["nombre_evento"].copy()
    mask_varias = n_pruebas_por_evento > 1
    nombre_carrera.loc[mask_varias] = (
        df_in.loc[mask_varias, "nombre_evento"] + " (" + df_in.loc[mask_varias, "nombre_prueba"] + ")"
    )
    return nombre_carrera


def derivar_distancia_km(df_in: pd.DataFrame) -> pd.Series:
    return pd.to_numeric(df_in["distancia_m"], errors="coerce") / 1000.0


_DIAS_SEMANA = ["Lunes", "Martes", "Miércoles", "Jueves", "Viernes", "Sábado", "Domingo"]


def derivar_fecha_y_dia_semana(df_in: pd.DataFrame) -> tuple[pd.Series, pd.Series]:
    fecha = pd.to_datetime(df_in["fecha"], errors="coerce")
    dia_semana = fecha.dt.dayofweek.map(dict(enumerate(_DIAS_SEMANA)))
    return fecha, dia_semana


def clasificar_tipo_modalidad(nombre) -> str:
    texto = "" if pd.isna(nombre) else nombre.lower()
    if re.search(r"esqu[ií]", texto):
        return "Otros"
    if re.search(r"duatl|triatl|triathlon|acuatl|aquathlon", texto):
        return "Multidisciplina"
    if re.search(r"\bbtt\b|\bmtb\b|\bbike\b|ciclis|ciclo", texto):
        return "Ciclismo y btt"
    if re.search(r"trail|cross|vertical|\bkv\b", texto):
        return "trail running"
    if re.search(r"marcha", texto):
        return "marcha"
    return "road running"


def clasificar_publico(nombre) -> str:
    texto = "" if pd.isna(nombre) else nombre.lower()
    if re.search(r"equipos?\b", texto):
        return "Equipos"
    if re.search(r"veteran|m[aá]ster|\bsenior\b", texto):
        return "Mayores/Veteranos"
    if re.search(r"\belite\b|[ée]lite|profesional", texto):
        return "Elite"
    if re.search(
        r"infantil|alev[ií]n|benjam|prebenjam|cadet|juvenil|j[uú]nior|menores|promesa|escolar|chupet|peques?\b",
        texto,
    ):
        return "Infantil"
    return "Absoluta/General"


### Aplicar las funciones al catálogo completo

In [3]:
df["nombre_carrera"] = construir_nombre_carrera(df)
df["distancia"] = derivar_distancia_km(df)
df["fecha"], df["dia_semana"] = derivar_fecha_y_dia_semana(df)
df["tipo_modalidad"] = df["nombre_carrera"].apply(clasificar_tipo_modalidad)
df["publico"] = df["nombre_carrera"].apply(clasificar_publico)

print(df["tipo_modalidad"].value_counts())
print()
print(df["publico"].value_counts())


tipo_modalidad
road running       2750
trail running       318
marcha               71
Ciclismo y btt        3
Multidisciplina       3
Name: count, dtype: int64

publico
Absoluta/General     1855
Infantil             1274
Elite                  10
Equipos                 5
Mayores/Veteranos       1
Name: count, dtype: int64


### Ubicación -- `municipio` directo, con respaldo por nombre cuando la web no lo trae

El propio `scrape_cronorunner.ipynb` ya extrae `municipio` directamente del icono de ubicación de cada ficha de evento en cronorunner.com, y esto funciona bien: no hay ningún valor nulo. Pero en **1.605 de las 3.145 filas reales (51 %, 636 eventos distintos)** esa extracción da el texto `"Sin Localidad definida"` -- que no es un fallo del scraper, sino el propio *placeholder* que muestra cronorunner.com cuando el organizador nunca rellenó la localidad (comprobado abriendo en directo varias de esas fichas antiguas).

Para recuperar localidad en esos casos, tiramos del nombre del evento: en cronorunner es muy habitual el patrón "número + VOLTA A PEU + POBLACIÓN" (p.ej. `"XX VOLTA A PEU CATADAU"` -> `Catadau`), así que aplicamos la misma heurística de aislar un candidato a lugar que ya usamos en `championsxip`/`youevent` (quitar numerales iniciales, año, paréntesis, palabras genéricas de carrera) **solo a los eventos sin localidad**, y lo geocodificamos con Nominatim. Si ni con el nombre se encuentra nada, la ubicación se queda en `NaN` -- no forzamos ningún valor inventado.

In [4]:
_RE_NUMERAL_INICIAL = re.compile(
    r"^\s*(?:[ivxlcdm]+|\d+)\s*[ºª]?\.?(?:er|do|ro|a)?\s*[-–]?\s+",
    re.IGNORECASE,
)
_RE_ANIO = re.compile(r"\b(19|20)\d{2}\b")
_RE_PARENTESIS = re.compile(r"\(.*?\)")
_RE_DISTANCIA_FINAL = re.compile(r"\s+\d+(?:[.,]\d+)?\s?k(?:m)?\.?$", re.IGNORECASE)

_PALABRAS_GENERICAS = {
    "carrera", "correr", "cross", "trail", "trailkids", "maraton", "marat\u00f3n", "mitja", "media", "medio",
    "milla", "popular", "populars", "urbana", "urbano", "semi", "escolar", "memorial", "trofeo",
    "circuito", "circuit", "campeonato", "duatlon", "duatl\u00f3n", "triatlon", "triatl\u00f3n",
    "nocturna", "nocturno", "solidaria", "solidario", "btt", "ruta", "gran", "premio", "fons", "fondo",
    "edicion", "edici\u00f3n", "subida", "pujada", "vuelta", "volta", "copa", "liga", "internacional",
    "provincial", "final", "comarcal", "nacional", "fase", "previa", "pedestre", "peu", "cursa",
    "ayuntamiento", "villa", "ciudad", "ciutat", "fiesta", "fiestas", "aniversario", "san", "sant",
    "silvestre", "vertical", "km", "k", "kv", "kilometro", "kil\u00f3metro", "legua", "travesía",
    "travesia", "travessia", "equipos", "equips", "y", "i", "gala",
}
_PREPOSICIONES = {"de", "del", "d\'", "a", "en", "al", "la", "les", "el", "els"}


def candidato_lugar(nombre: str) -> str:
    """Aísla un candidato a población dentro del nombre de un evento, quitando
    numerales, año, paréntesis y palabras genéricas de carrera por delante."""
    if not isinstance(nombre, str) or not nombre.strip():
        return ""

    texto = nombre.strip()
    texto = _RE_NUMERAL_INICIAL.sub("", texto)
    texto = _RE_ANIO.split(texto)[0]
    texto = _RE_PARENTESIS.sub("", texto)
    texto = _RE_DISTANCIA_FINAL.sub("", texto)
    texto = texto.strip(" -–,.")

    tokens = texto.split()
    i = 0
    while i < len(tokens):
        tok = tokens[i].lower().strip(",.-\'\u2019\"")
        if tok in _PALABRAS_GENERICAS or tok in _PREPOSICIONES or re.fullmatch(r"\d+([.,]\d+)?k?m?", tok):
            i += 1
        else:
            break

    candidato = " ".join(tokens[i:]).strip(" ,.-–")
    return candidato or texto


# El placeholder de la web no es una localidad real: lo tratamos como nulo.
df.loc[df["municipio"] == "Sin Localidad definida", "municipio"] = pd.NA

_mask_sin_municipio = df["municipio"].isna()
_candidatos = df.loc[_mask_sin_municipio, "nombre_evento"].drop_duplicates().apply(candidato_lugar)
_n_eventos_sin_municipio = df.loc[_mask_sin_municipio, "nombre_evento"].nunique()
print(f"{_mask_sin_municipio.sum()} filas sin municipio directo ({_n_eventos_sin_municipio} eventos distintos)")
print("Ejemplos nombre_evento -> candidato_lugar:")
print(
    pd.DataFrame({
        "nombre_evento": df.loc[_mask_sin_municipio, "nombre_evento"].drop_duplicates(),
        "candidato_lugar": _candidatos,
    }).sample(min(15, len(_candidatos)), random_state=0)
)


1605 filas sin municipio directo (632 eventos distintos)
Ejemplos nombre_evento -> candidato_lugar:
                                          nombre_evento  \
2176                        XXX GRAN FONS DE MASSANASSA   
2890                   XXXV BESTDRIVE 10K. MISLATA 2018   
3128            X MITJA MARATO i X10K CIUTAT DE TORRENT   
2041              VII VOLTA A PEU SANTA CECILIA CULLERA   
3132  II CARRERA SOLIDARIA AFACAM 10K PUERTO DE SAGUNTO   
1835                           III 10k POBLA DE FARNALS   
2068                     XXXI 10K i 20K CIUTAT DE SUECA   
2865        IX CARRERA POPULAR PARQUE NATURAL DEL TÚRIA   
2797                       XXIX VOLTA A PEU A L´ALCÚDIA   
2242                    IV 10MILPASSOS KANGURS BENIFAIÓ   
1911                             XXI GRAN FONS DE PUÇOL   
1752                          XXXII VOLTA A PEU A TURÍS   
2222     XXXI VOLTA A PEU FIBRA VALENCIA VILA D'ALAQUAS   
2255                          XVIII VOLTA A PEU GAVARDA   
2350           

Geocodificamos en dos tandas -- los municipios reales tal cual, y los candidatos sacados del nombre con el mismo reintento progresivo (probar la frase completa y, si no hay resultado, ir quitando palabras por delante) que usamos en `youevent`/`championsxip`, porque los candidatos aquí son más ruidosos. Los que no encuentren nada (ni municipio real ni candidato) se quedan en `NaN`, como ha pedido Clàudia -- sin inventar ningún valor. Se guarda un checkpoint cada 25 consultas en `cronorunner_ubicaciones.csv` para poder retomar si se corta.

In [5]:
from pathlib import Path
import time as _time
import certifi
import ssl
from geopy.geocoders import Nominatim
from geopy.exc import GeopyError

RUTA_UBICACIONES = Path("../../data/raw/cronorunner/cronorunner_ubicaciones.csv")

# El Python de python.org en este Mac no trae un almacén de certificados de
# sistema enlazado, así que geopy falla con SSLCertVerificationError sin esto.
_SSL_CONTEXT = ssl.create_default_context(cafile=certifi.where())


def _geocode_con_reintentos(geolocalizador, candidato, pausa_segundos):
    tokens = candidato.split()
    max_inicio = max(0, len(tokens) - 2) if len(tokens) > 1 else 0
    for inicio in range(max_inicio + 1):
        query = " ".join(tokens[inicio:])
        if not query:
            break
        try:
            ubic = geolocalizador.geocode(
                f"{query}, España", exactly_one=True, country_codes="es",
                addressdetails=True, timeout=10,
            )
        except GeopyError:
            ubic = None
        if ubic:
            return ubic
        if inicio < max_inicio:
            _time.sleep(pausa_segundos)
    return None


def geocodificar_ubicaciones_cronorunner(df_in, pausa_segundos=1.0):
    geolocalizador = Nominatim(user_agent="tfm_carreras_populares_es", ssl_context=_SSL_CONTEXT)

    filas = []
    if RUTA_UBICACIONES.exists():
        filas = pd.read_csv(RUTA_UBICACIONES, dtype=str).to_dict("records")
    hechas = {f["clave"] for f in filas}

    mask_real = df_in["municipio"].notna()
    claves_reales = df_in.loc[mask_real, "municipio"].drop_duplicates().tolist()
    claves_candidatas = df_in.loc[~mask_real, "nombre_evento"].drop_duplicates().tolist()

    campos = ["clave", "query", "municipio", "comarca", "provincia", "lat", "lon"]

    def _resolver(clave, query, i, total):
        if clave in hechas:
            return
        fila = {c: None for c in campos}
        fila["clave"], fila["query"] = clave, query
        ubic = _geocode_con_reintentos(geolocalizador, query, pausa_segundos)
        if ubic:
            direccion = ubic.raw.get("address", {})
            fila["municipio"] = (
                direccion.get("city") or direccion.get("town") or direccion.get("village")
                or direccion.get("municipality")
            )
            fila["comarca"] = direccion.get("county") or direccion.get("state_district")
            fila["provincia"] = direccion.get("province") or direccion.get("state")
            fila["lat"], fila["lon"] = ubic.latitude, ubic.longitude
        filas.append(fila)
        hechas.add(clave)
        if i % 25 == 0:
            pd.DataFrame(filas).to_csv(RUTA_UBICACIONES, index=False)
            print(f"  [{i}/{total}] checkpoint guardado")
        _time.sleep(pausa_segundos)

    total = len(claves_reales) + len(claves_candidatas)
    for i, municipio in enumerate(claves_reales, 1):
        _resolver(municipio, municipio, i, total)
    for i, nombre in enumerate(claves_candidatas, 1):
        _resolver(nombre, candidato_lugar(nombre), len(claves_reales) + i, total)

    df_ubicaciones = pd.DataFrame(filas).drop_duplicates("clave")
    df_ubicaciones.to_csv(RUTA_UBICACIONES, index=False)
    return df_ubicaciones


df_ubicaciones_cronorunner = geocodificar_ubicaciones_cronorunner(df)

# clave de fusión: el municipio real si lo hay, si no el nombre del evento
# (que es la clave que se usó para geocodificar el candidato sacado del nombre)
df["_clave_ubicacion"] = df["municipio"].where(df["municipio"].notna(), df["nombre_evento"])
df = df.merge(
    df_ubicaciones_cronorunner[["clave", "municipio", "comarca", "provincia"]]
        .rename(columns={"clave": "_clave_ubicacion", "municipio": "_municipio_resuelto"}),
    on="_clave_ubicacion", how="left",
)
df["municipio"] = df["_municipio_resuelto"].combine_first(df["municipio"])
df = df.drop(columns=["_clave_ubicacion", "_municipio_resuelto"])

print(f"municipio resuelto: {df['municipio'].notna().sum()}/{len(df)}")
print(f"comarca resuelta: {df['comarca'].notna().sum()}/{len(df)}")
print(f"provincia resuelta: {df['provincia'].notna().sum()}/{len(df)}")


  [25/754] checkpoint guardado
  [50/754] checkpoint guardado
  [75/754] checkpoint guardado
  [100/754] checkpoint guardado
  [125/754] checkpoint guardado
  [150/754] checkpoint guardado
  [175/754] checkpoint guardado
  [200/754] checkpoint guardado
  [225/754] checkpoint guardado
  [250/754] checkpoint guardado
  [275/754] checkpoint guardado
  [300/754] checkpoint guardado
  [325/754] checkpoint guardado
  [350/754] checkpoint guardado
  [375/754] checkpoint guardado
  [400/754] checkpoint guardado
  [425/754] checkpoint guardado
  [450/754] checkpoint guardado
  [475/754] checkpoint guardado
  [500/754] checkpoint guardado
  [525/754] checkpoint guardado
  [550/754] checkpoint guardado
  [575/754] checkpoint guardado
  [600/754] checkpoint guardado
  [625/754] checkpoint guardado
  [650/754] checkpoint guardado
  [675/754] checkpoint guardado
  [700/754] checkpoint guardado
  [725/754] checkpoint guardado
  [750/754] checkpoint guardado
municipio resuelto: 2796/3145
comarca resue

### Esquema común entre las fuentes

In [6]:
df["fuente"] = "cronorunner"

_COLUMNAS_COMUNES = [
    "fuente", "nombre_carrera", "fecha", "dia_semana", "distancia", "tipo_modalidad", "publico",
    "finisher_d", "finisher_h", "municipio", "comarca", "provincia",
]
_COLUMNAS_PROPIAS = [c for c in df.columns if c not in _COLUMNAS_COMUNES]
df = df[_COLUMNAS_COMUNES + _COLUMNAS_PROPIAS]

print(df.shape)
df.head(8)


(3145, 17)


,fuente,nombre_carrera,fecha,dia_semana,distancia,tipo_modalidad,publico,finisher_d,finisher_h,municipio,comarca,provincia,evento_ref,resultado_ref,nombre_evento,nombre_prueba,distancia_m
0,cronorunner,XXII VOLTA A PEU LORIGUILLA (Junior (2009-2010)),2026-09-04,Viernes,0.00,road running,Infantil,NaN,NaN,Loriguilla,València / Valencia,Comunitat Valenciana,448-2212,448-2212-3895,XXII VOLTA A PEU LORIGUILLA,Junior (2009-2010),0
1,cronorunner,XXII VOLTA A PEU LORIGUILLA (Benjamin (2017-20...,2026-09-04,Viernes,0.30,road running,Infantil,9.0,8.0,Loriguilla,València / Valencia,Comunitat Valenciana,448-2212,448-2212-3861,XXII VOLTA A PEU LORIGUILLA,Benjamin (2017-2018),300
2,cronorunner,XXII VOLTA A PEU LORIGUILLA (Alevin (2015-2016)),2026-09-04,Viernes,0.70,road running,Infantil,5.0,7.0,Loriguilla,València / Valencia,Comunitat Valenciana,448-2212,448-2212-3862,XXII VOLTA A PEU LORIGUILLA,Alevin (2015-2016),700
3,cronorunner,XXII VOLTA A PEU LORIGUILLA (Infantil (2013-20...,2026-09-04,Viernes,1.05,road running,Infantil,2.0,1.0,Loriguilla,València / Valencia,Comunitat Valenciana,448-2212,448-2212-3863,XXII VOLTA A PEU LORIGUILLA,Infantil (2013-2014),1050
4,cronorunner,XXII VOLTA A PEU LORIGUILLA (Cadete (2011-2012)),2026-09-04,Viernes,1.40,road running,Infantil,1.0,2.0,Loriguilla,València / Valencia,Comunitat Valenciana,448-2212,448-2212-3864,XXII VOLTA A PEU LORIGUILLA,Cadete (2011-2012),1400
5,cronorunner,XXII VOLTA A PEU LORIGUILLA (10k),2026-09-04,Viernes,10.00,road running,Absoluta/General,58.0,220.0,Loriguilla,València / Valencia,Comunitat Valenciana,448-2212,448-2212-3694,XXII VOLTA A PEU LORIGUILLA,10k,10000
6,cronorunner,XXIX VOLTA A PEU LA CANYADA (Sub 10 (Nacidos e...,2026-08-29,Sábado,1.00,road running,Absoluta/General,NaN,NaN,Paterna,l'Horta Nord,Comunitat Valenciana,331-2261,331-2261-3814,XXIX VOLTA A PEU LA CANYADA,Sub 10 (Nacidos en 2017 y posteriores),1000
7,cronorunner,XXIX VOLTA A PEU LA CANYADA (Sub 14 (Nacidos e...,2026-08-29,Sábado,1.00,road running,Absoluta/General,NaN,NaN,Paterna,l'Horta Nord,Comunitat Valenciana,331-2261,331-2261-3815,XXIX VOLTA A PEU LA CANYADA,Sub 14 (Nacidos entre 2013 y 2016),1000


In [7]:
RUTA_LIMPIO = "../../data/processed/cronorunner/DF_CRONORUNNER_LIMPIO.csv"
df.to_csv(RUTA_LIMPIO, index=False)
print("Guardado:", RUTA_LIMPIO, df.shape)


Guardado: ../../data/processed/cronorunner/DF_CRONORUNNER_LIMPIO.csv (3145, 17)


### Prueba con una muestra real pequeña

Antes de fiarnos del resultado con el catálogo completo, probamos las mismas funciones (`construir_nombre_carrera`, `derivar_distancia_km`, `derivar_fecha_y_dia_semana`, `clasificar_tipo_modalidad`, `clasificar_publico`) contra una muestra de 12 filas con datos reales, comprobados a mano en el navegador: las 6 pruebas de "XXII Volta a peu Loriguilla" (viernes 04/09/2026) y las 6 de "XXXIV Volta a peu nocturna a Manuel" (viernes 28/08/2026) -- los mismos dos eventos usados para probar el scraper. `finisher_h`/`finisher_d` llevan los valores reales para las 2 pruebas que hemos comprobado a mano (10k y Benjamín de Loriguilla, 7k de Manuel); el resto se deja en blanco porque no hemos abierto esas páginas de resultado en concreto.

In [8]:
_df_prueba = pd.read_csv(
    "../../data/raw/cronorunner/muestra_limpieza/DF_CRONORUNNER_SUCIO_muestra.csv",
    dtype={"evento_ref": str, "resultado_ref": str},
)

_df_prueba["nombre_carrera"] = construir_nombre_carrera(_df_prueba)
_df_prueba["distancia"] = derivar_distancia_km(_df_prueba)
_df_prueba["fecha"], _df_prueba["dia_semana"] = derivar_fecha_y_dia_semana(_df_prueba)
_df_prueba["tipo_modalidad"] = _df_prueba["nombre_carrera"].apply(clasificar_tipo_modalidad)
_df_prueba["publico"] = _df_prueba["nombre_carrera"].apply(clasificar_publico)

_esperado_publico = {
    "XXII VOLTA A PEU LORIGUILLA (10k)": "Absoluta/General",
    "XXII VOLTA A PEU LORIGUILLA (Benjamin (2017-2018))": "Infantil",
    "XXII VOLTA A PEU LORIGUILLA (Alevin (2015-2016))": "Infantil",
    "XXII VOLTA A PEU LORIGUILLA (Infantil (2013-2014))": "Infantil",
    "XXII VOLTA A PEU LORIGUILLA (Cadete (2011-2012))": "Infantil",
    "XXII VOLTA A PEU LORIGUILLA (Junior (2009-2010))": "Infantil",
    "XXXIV VOLTA A PEU NOCTURNA A MANUEL (7k)": "Absoluta/General",
    "XXXIV VOLTA A PEU NOCTURNA A MANUEL (Prebenjamin (2019-2020))": "Infantil",
    "XXXIV VOLTA A PEU NOCTURNA A MANUEL (Benjamin (2017-2018))": "Infantil",
    "XXXIV VOLTA A PEU NOCTURNA A MANUEL (Alevin (2015-2016))": "Infantil",
    "XXXIV VOLTA A PEU NOCTURNA A MANUEL (Infantil (2013-2014))": "Infantil",
    "XXXIV VOLTA A PEU NOCTURNA A MANUEL (Cadet (2009-2012))": "Infantil",
}
assert _df_prueba.shape[0] == 12
assert (_df_prueba["tipo_modalidad"] == "road running").all(), "se esperaba road running en las 12 filas"
for nombre, publico_esperado in _esperado_publico.items():
    fila = _df_prueba.loc[_df_prueba["nombre_carrera"] == nombre]
    assert len(fila) == 1, f"no encuentro la fila de {nombre!r}"
    publico_real = fila["publico"].iloc[0]
    assert publico_real == publico_esperado, f"{nombre!r}: publico={publico_real!r}, esperado {publico_esperado!r}"

assert _df_prueba.loc[_df_prueba["nombre_carrera"].str.contains("10k"), "distancia"].iloc[0] == 10.0
assert _df_prueba.loc[_df_prueba["nombre_carrera"].str.contains("7k"), "distancia"].iloc[0] == 7.0
assert _df_prueba.loc[_df_prueba["nombre_evento"].str.contains("LORIGUILLA"), "dia_semana"].iloc[0] == "Viernes"
assert _df_prueba.loc[_df_prueba["nombre_evento"].str.contains("MANUEL"), "dia_semana"].iloc[0] == "Viernes"

print("Todo OK: las funciones de limpieza se comportan bien contra la muestra real.")
_df_prueba[["nombre_carrera", "distancia", "dia_semana", "tipo_modalidad", "publico", "finisher_h", "finisher_d"]]


Todo OK: las funciones de limpieza se comportan bien contra la muestra real.


,nombre_carrera,distancia,dia_semana,tipo_modalidad,publico,finisher_h,finisher_d
0,XXII VOLTA A PEU LORIGUILLA (10k),10.00,Viernes,road running,Absoluta/General,220.0,58.0
1,XXII VOLTA A PEU LORIGUILLA (Benjamin (2017-20...,0.30,Viernes,road running,Infantil,8.0,9.0
2,XXII VOLTA A PEU LORIGUILLA (Alevin (2015-2016)),0.70,Viernes,road running,Infantil,NaN,NaN
3,XXII VOLTA A PEU LORIGUILLA (Infantil (2013-20...,1.05,Viernes,road running,Infantil,NaN,NaN
4,XXII VOLTA A PEU LORIGUILLA (Cadete (2011-2012)),1.40,Viernes,road running,Infantil,NaN,NaN
5,XXII VOLTA A PEU LORIGUILLA (Junior (2009-2010)),0.00,Viernes,road running,Infantil,NaN,NaN
6,XXXIV VOLTA A PEU NOCTURNA A MANUEL (7k),7.00,Viernes,road running,Absoluta/General,363.0,135.0
7,XXXIV VOLTA A PEU NOCTURNA A MANUEL (Prebenjam...,0.20,Viernes,road running,Infantil,NaN,NaN
8,XXXIV VOLTA A PEU NOCTURNA A MANUEL (Benjamin ...,0.55,Viernes,road running,Infantil,NaN,NaN
9,XXXIV VOLTA A PEU NOCTURNA A MANUEL (Alevin (2...,1.10,Viernes,road running,Infantil,NaN,NaN
